# AudioGen - Generation de Bruitages par IA

**Module :** 02-Audio-Advanced
**Niveau :** Intermediaire
**Prerequis :** bases du traitement du signal audio ([serie Foundation](../01-Foundation/README.md)), vocabulaire de la generation par IA
**Temps estime :** 30-45 minutes (GPU), nettement plus sur CPU

| Information | Valeur |
|---|---|
| Modele | `facebook/audiogen-medium` (Kreuk et al. 2023) |
| Librairie | `audiocraft` (Meta AI) |
| Tache | Text-to-audio : scenes sonores et bruitages decrits en texte |
| Sortie | WAV 16 kHz mono |
| Materiel | GPU NVIDIA recommande (8 Go suffisent) ; repli CPU documente mais lent |

Ce carnet est le volet **text-to-audio autoregressif** de la serie : la [sous-serie 02](README.md) couvrait la musique (MusicGen, MIDI, YuE2, ACE-Step), la [sous-serie 06](../06-Diffusion-SOTA/06-1-AudioLDM-SOTA-Comparison.ipynb) couvre le text-to-audio par diffusion (AudioLDM-S). AudioGen occupe la troisieme case : un modele autoregressif qui ecrit l'audio comme un langage.

**Navigation :** [<< 02-10](Audio-02-10-Effets-Sonores-AudioLDM2-Python.ipynb) | [Index](../README.md) | [Suivant >>](../03-Orchestration/README.md)

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"

# Parametres AudioGen
model_size = "facebook/audiogen-medium"   # unique variante publiee
duration_seconds = 6                      # Duree de generation (secondes, 5-10 conseille)
device = "cuda"                           # "cuda" ou "cpu"
temperature = 1.0                         # Temperature de sampling (0.5-1.5)
top_k = 250                               # Nucleus top-k de sampling
cfg_coef = 3.0                            # Guidance sans classifieur (gamma papier : 3.0)

# Configuration
generate_audio = True              # Generer les fichiers audio
save_results = True                # Sauvegarder les fichiers generes
sweep_full = True                  # Balayage complet temperature/top_k (GPU conseille)


### Parametres Papermill et configuration

Cette cellule definit les parametres modifiables par Papermill (`notebook_mode`, `model_size`, `duration_seconds`, `temperature`, `top_k`, `cfg_coef`). En execution batch, la cellule `# Parameters` ci-dessous ecrase ces valeurs sans toucher au source : c'est le meme contrat que les carnets [MusicGen](02-3-MusicGen-Generation.ipynb) et [AudioLDM](../06-Diffusion-SOTA/06-1-AudioLDM-SOTA-Comparison.ipynb).

In [2]:
# Parameters
notebook_mode = "batch"
skip_widgets = "true"


## Introduction : generer l'audio comme un langage

AudioGen (Kreuk et al., ICLR 2023) resout le text-to-audio par une chaine en **deux etages** :

1. un **codec neural** (famille EnCodec) compresse l'audio brut en une sequence **discrete** de tokens a 16 kHz ;
2. un **modele de langage autoregressif** (Transformer decodeur) ecrit la sequence de tokens, conditionne par un encodeur de texte **T5** pre-entraine.

Le papier entraine deux tailles : **285 M** de parametres (hidden 768, 24 couches, 16 tetes, T5-base) et **1 Mds** (hidden 1280, 36 couches, 20 tetes, T5-large). L'entrainement curationne **10 jeux de donnees** (AudioSet, AudioCaps, Clotho v2, VGG-Sound, FSD50K, BBC Sound Effects, Sonniss GDC, entre autres) et applique une augmentation par **melange d'extraits** qui force le modele a separer des sources concurrentes. Sur AudioCaps, la variante 1 Md atteint un **FAD de 1,82** contre 7,39 pour DiffSound (400 M) -- et une relevance au texte (REL) de 68,73 contre 55,91.

**Pourquoi un troisieme modele alors que la sous-serie 06 fait deja du text-to-audio ?** Parce que le paradigme n'est pas le meme. AudioLDM-S (06-1) est une **diffusion latente** : rapide par pas parallelisables, forte sur les textures. AudioGen est **autoregressif** : chaque seconde d'audio est ecrite token par token, ce qui coute en latence mais achete une composition multi-concepts plus docile -- le prompt du papier, « a dog barks while somebody plays the trumpet in a busy street », empile trois sources que le modele n'a jamais vues ensemble dans les donnees. Ce carnet mesure les deux faces de ce compromis sur la machine disponible.

Les parametres Papermill definissent la duree des generations (`duration_seconds`, 5-10 s), la temperature et le top-k de sampling, et le coefficient de guidance `cfg_coef` (le papier utilise un gamma de 3,0). La cellule suivante prepare l'environnement : imports, helpers audio partages et verification GPU.

In [3]:
# Setup environnement et imports
import os
import sys
import json
import time
import gc
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', message=r'.*weight_norm.*is deprecated.*')
warnings.filterwarnings('ignore', message='.*Torch was not compiled with flash attention.*')
from datetime import datetime
from typing import Dict, List, Any, Optional
import logging

import numpy as np
from IPython.display import Audio, display, HTML

# Import helpers GenAI
GENAI_ROOT = Path.cwd()
while GENAI_ROOT.name != 'GenAI' and len(GENAI_ROOT.parts) > 1:
    GENAI_ROOT = GENAI_ROOT.parent

HELPERS_PATH = GENAI_ROOT / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.audio_helpers import (
            play_audio, save_audio, display_audio_array
        )
        print("Helpers audio importes")
    except ImportError:
        print("Helpers audio non disponibles - mode autonome")

# Repertoires
OUTPUT_DIR = GENAI_ROOT / 'outputs' / 'audio' / 'audiogen'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration logging
logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('audiogen')
# Les dependances tierces emettent au niveau INFO (requetes HTTP, decoupage en
# codebooks) et xformers signale l'absence de Triton par une trace : rien de
# tout cela n'apprend quelque chose sur ce que fait le carnet, et une trace
# dans une sortie committee se lit a tort comme un echec.
for _bavard in ('httpx2', 'audiocraft', 'xformers'):
    logging.getLogger(_bavard).setLevel(logging.WARNING)
logging.getLogger('xformers').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', message='IProgress not found')

# Verification GPU
gpu_available = False
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU : {gpu_name} ({gpu_vram:.1f} GB VRAM)")
    else:
        print("GPU non disponible - AudioGen sera tres lent sur CPU")
        if device == "cuda":
            device = "cpu"
            print("Fallback vers CPU")
except ImportError:
    print("torch non installe")
    device = "cpu"

print(f"\nAudioGen - Generation de Bruitages par IA")
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode : {notebook_mode}, Device : {device}")
print(f"Modele : {model_size}, Duree : {duration_seconds}s")
print(f"Temperature : {temperature}, top_k : {top_k}, cfg_coef : {cfg_coef}")
print(f"Sortie : GenAI/{OUTPUT_DIR.relative_to(GENAI_ROOT)}")


Helpers audio importes


GPU : NVIDIA GeForce RTX 3070 Laptop GPU (8.0 GB VRAM)

AudioGen - Generation de Bruitages par IA
Date : 2026-11-14 22:20:15
Mode : batch, Device : cuda
Modele : facebook/audiogen-medium, Duree : 6s
Temperature : 1.0, top_k : 250, cfg_coef : 3.0
Sortie : GenAI/outputs\audio\audiogen


### Interpretation : configuration de l'environnement

L'environnement est configure. La ligne GPU indique le materiel qui portera les mesures de ce carnet (RTF, memoire) : ces nombres sont des ordres de grandeur pour ce materiel, pas des constantes du modele -- le [carnet AudioLDM](../06-Diffusion-SOTA/06-1-AudioLDM-SOTA-Comparison.ipynb) fait la meme reserve pour ses propres latences.

AudioGen-medium est un modele Meta soumis a une licence propre sur Hugging Face : la cellule suivante charge le fichier `.env` de la serie GenAI pour recuperer le token HF (`HUGGINGFACE_TOKEN`) qui autorise le telechargement des poids.

In [4]:
# Chargement robuste de la configuration .env
from dotenv import load_dotenv
import os
# Recherche du .env dans tous les parents (pour Papermill qui change le cwd)
current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.parent.name}/{env_path.name}")
        env_loaded = True
        break
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
    current_path = current_path.parent
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")

# huggingface_hub lit HF_TOKEN (puis HUGGING_FACE_HUB_TOKEN), jamais HUGGINGFACE_TOKEN :
# on mappe la cle du .env vers le nom attendu, sans jamais afficher la valeur.
if os.getenv("HUGGINGFACE_TOKEN") and not os.getenv("HF_TOKEN"):
    os.environ["HF_TOKEN"] = os.environ["HUGGINGFACE_TOKEN"]

hf_token_present = bool(os.getenv("HF_TOKEN"))
print(f"Token HF : {'present' if hf_token_present else 'absent (modeles publics uniquement)'}")


.env charge depuis: GenAI/.env
Token HF : present


## Dependances GPU Optionnelles

Ce notebook utilise un modele GPU optionnel. Pour l'activer : verifier que `audiocraft` est installe dans l'environnement Python du kernel (`pip install audiocraft`), ainsi que `torch` avec le backend CUDA correspondant au pilote. Sur Windows, `audiocraft` exige une operation d'attention efficace dans `xformers` : la paire mesuree qui fournit ces noyaux est `torch==2.7.0` + `xformers==0.0.30` (les versions plus recentes de `xformers` deportent ces operations vers `mslk`, sans wheel Windows fonctionnel a ce jour). Sans GPU, le carnet s'execute quand meme (repli CPU documente) mais chaque generation se compte en minutes : les sorties commitees de ce carnet proviennent d'une execution GPU.

## Section 1 : Presentation d'AudioGen et chargement du modele

Le modele publie `facebook/audiogen-medium` correspond a la lignee du papier (decodeur autoregressif sur tokens de codec, conditionnement T5). La cellule suivante le charge et mesure ce que ce chargement coute : temps, parametres, memoire. Les nombres qui nous interessent pour la suite sont le **RTF** (real-time factor : secondes de calcul par seconde d'audio produite) et le **pic memoire** -- les deux grandeurs que le [06-1](../06-Diffusion-SOTA/06-1-AudioLDM-SOTA-Comparison.ipynb) mesure aussi sur AudioLDM, et qui serviront a la comparaison de la section 4.

In [5]:
# Chargement du modele AudioGen
print("CHARGEMENT DU MODELE AUDIOGEN")
print("=" * 45)

audiogen_loaded = False
model = None
load_time = None
t0 = time.time()
try:
    from audiocraft.models import AudioGen
    # Les poids du fichier sont en fp32 (environ 7 Go). Les relire d'un bloc les
    # materialise en RAM en plus du modele : sur une machine a 16 Go, Windows
    # peut refuser l'allocation (erreur 1455, fichier de pagination
    # insuffisant). mmap=True lit les tenseurs depuis le fichier a la demande,
    # si bien que la memoire ne porte que le modele effectivement construit.
    _torch_load = torch.load
    torch.load = lambda *a, **kw: _torch_load(*a, **{**kw, "mmap": True})
    try:
        # Le chargeur d'audiocraft construit le LM en float16 des que device
        # n'est pas "cpu" : inutile de passer par un chargement CPU fp32, le
        # modele tient sur un GPU de 8 Go tel quel.
        model = AudioGen.get_pretrained(model_size, device=device)
    finally:
        torch.load = _torch_load
    load_time = time.time() - t0
    audiogen_loaded = True
except Exception as e:
    print(f"ECHEC chargement : {type(e).__name__}")
    print(str(e)[:300])
    print("Le carnet continue en mode documentaire : les sections suivantes seront sautees.")

# Les mesures qui suivent sont hors du bloc de chargement : une erreur de
# diagnostic ne doit pas pouvoir s'afficher comme un echec du modele.
if audiogen_loaded:
    # Precision reelle, mesuree et non supposee : le chargeur met le LM en
    # float16 sauf quelques tenseurs fp32, et le modele de compression reste
    # en fp32.
    precisions = {}
    for p in model.lm.parameters():
        nom = str(p.dtype).replace("torch.", "")
        precisions[nom] = precisions.get(nom, 0) + p.numel()
    compute_dtype = max(precisions, key=precisions.get)
    # Recensement des parametres par composant
    n_lm = sum(p.numel() for p in model.lm.parameters())
    # sample_rate est une propriete en lecture seule, heritee du modele de
    # compression : on la lit, on ne l'assigne pas.
    print(f"Modele charge en {load_time:.1f}s sur {device}")
    print(f"Parametres du modele de langage : {n_lm:,}")
    print(f"Frequence d'echantillonnage : {model.sample_rate} Hz")
    print(f"Precision dominante du LM : {compute_dtype}")
    for nom, n in sorted(precisions.items(), key=lambda kv: -kv[1]):
        print(f"  {nom} : {n:,} parametres")

    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        vram_alloc = torch.cuda.memory_allocated() / (1024**3)
        print(f"VRAM allouee apres chargement : {vram_alloc:.2f} GiB")

audiogen_loaded


CHARGEMENT DU MODELE AUDIOGEN


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Loading weights:   4%|▎         | 7/195 [00:00<00:03, 57.80it/s]

Loading weights:   7%|▋         | 13/195 [00:00<00:06, 27.66it/s]

Loading weights: 100%|██████████| 195/195 [00:00<00:00, 436.53it/s]

Modele charge en 15.0s sur cuda
Parametres du modele de langage : 1,839,131,136
Frequence d'echantillonnage : 16000 Hz
Precision dominante du LM : float16
  float16 : 1,812,381,696 parametres
  float32 : 26,749,440 parametres
VRAM allouee apres chargement : 4.94 GiB


True

### Interpretation : chargement du modele

Trois ordres de grandeur a retenir de cette sortie : le **nombre de parametres du decodeur** (a comparer aux 285 M / 1 Md du papier et aux 421 M de l'AudioLDM-S de 06-1), le **temps de chargement** (domine par le premier telechargement des poids, puis par la lecture du cache), et la **memoire allouee** -- un modele de 1,5 milliard de parametres annonce (1,84 milliard comptes dans la cellule ci-dessus, conditionneur texte compris) tient en demi-precision dans quelques GiB, ce qui le rend viable sur un GPU de bureau de 8 Go.

## Section 2 : Generation text-to-audio de scenes sonores

La generation est le coeur du carnet : quatre **scenes sonores bornees** (6 s chacune) choisies pour couvrir les regimes difficiles du text-to-audio --

1. **un transitoire sec** (*un marteau frappe une surface en bois, suivi d'un silence echoant*) : le regime ou la diffusion de 06-1 decroche le plus (distance mel ~41 dB sur la classe *dog*) ;
2. **une texture continue** (*des oiseaux dans une foret calme avec un leger vent*) : le regime ou la diffusion est forte ;
3. **une composition multi-concepts** (*un chien aboie pendant que quelqu'un joue de la trompette dans une rue animee*) : l'exemple du papier, trois sources concurrentes ;
4. **une scene superposee** (*la pluie sur un toit de metal avec un tonnerre lointain*) : fond regulier + evenement ponctuel.

Chaque clip est sauvegarde en WAV dans le repertoire de sortie ; seul un **apercu de 3 s** est embarque dans le carnet (anti-bloat), le clip complet restant local.

In [6]:
# Generation des scenes sonores
print("GENERATION TEXT-TO-AUDIO")
print("=" * 45)

demo_prompts = [
    "A hammer hitting a wooden surface, followed by echoing silence",
    "Birds singing in a quiet forest with light wind",
    "A dog barks while somebody plays the trumpet in a busy street",
    "Rain falling on a metal roof with distant thunder",
]

generation_stats = []

def audio_descriptors(wav_np, sr):
    """Descripteurs simples : RMS, centroide spectral, platitude spectrale."""
    x = wav_np.astype(np.float64)
    rms = float(np.sqrt(np.mean(x ** 2)))
    frame, hop = 1024, 512
    n_frames = max(1, (len(x) - frame) // hop + 1)
    window = np.hanning(frame)
    mags = np.array([
        np.abs(np.fft.rfft(x[i * hop:i * hop + frame] * window))
        for i in range(n_frames)
    ])
    freqs = np.fft.rfftfreq(frame, 1.0 / sr)
    mag_sum = mags.sum(axis=1) + 1e-12
    # Moyenne ponderee des frequences par l'amplitude : sans la division par
    # mag_sum on obtient une somme, qui croit avec la duree de l'analyse et
    # depasse la frequence de Nyquist.
    centroid = float(((mags * freqs).sum(axis=1) / mag_sum).mean())
    # Platitude spectrale : moyenne geometrique / moyenne arithmetique sur les
    # BINS de frequence de chaque trame, puis moyennee dans le temps. Moyenner
    # dans le temps par bin mesurerait la constance temporelle, pas la
    # platitude du spectre (un sinus pur y rendrait 1,0).
    geometric = np.exp(np.log(mags + 1e-12).mean(axis=1))
    arithmetic = mags.mean(axis=1) + 1e-12
    flatness = float((geometric / arithmetic).mean())
    return {"rms": rms, "centroid_hz": centroid, "flatness": flatness}

if audiogen_loaded and generate_audio:
    model.set_generation_params(
        duration=duration_seconds,
        temperature=temperature,
        top_k=top_k,
        cfg_coef=cfg_coef,
    )
    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    t_all = time.time()
    wavs = model.generate(demo_prompts)
    gen_time_all = time.time() - t_all
    total_audio = len(demo_prompts) * duration_seconds
    rtf_batch = gen_time_all / total_audio

    print(f"{len(demo_prompts)} scenes x {duration_seconds}s generees en {gen_time_all:.1f}s")
    print(f"RTF (fournie, batch de {len(demo_prompts)}) : {rtf_batch:.3f}")
    if device == "cuda" and torch.cuda.is_available():
        peak = torch.cuda.max_memory_allocated() / (1024**3)
        print(f"Pic VRAM : {peak:.2f} GiB")

    import torchaudio
    for i, (prompt, wav) in enumerate(zip(demo_prompts, wavs)):
        # Les sorties du modele sont en demi-precision : conversion explicite
        # en float32 avant sauvegarde et analyse des descripteurs.
        w = wav.detach().cpu().float()
        fname = f"scene_{i}.wav"
        if save_results:
            torchaudio.save(str(OUTPUT_DIR / fname), w, model.sample_rate)
        wav_np = w.numpy().squeeze()
        d = audio_descriptors(wav_np, model.sample_rate)
        gen_time_one = gen_time_all / len(demo_prompts)
        generation_stats.append({"prompt": prompt, "fichier": fname, **d})
        print(f"\nScene {i+1} : {prompt}")
        print(f"  fichier : {fname} | {len(wav_np)/model.sample_rate:.2f}s | "
              f"RMS {d['rms']:.4f} | centroide {d['centroid_hz']:.0f} Hz | platitude {d['flatness']:.3f}")
        # Apercu ~3 s embarque (clip complet sauvegarde localement, anti-bloat)
        n_prev = min(len(wav_np), model.sample_rate * 3)
        display(Audio(wav_np[:n_prev], rate=model.sample_rate))
else:
    print("Generation sautee : modele non charge ou generate_audio=False")


GENERATION TEXT-TO-AUDIO


4 scenes x 6s generees en 27.1s
RTF (fournie, batch de 4) : 1.129
Pic VRAM : 5.66 GiB

Scene 1 : A hammer hitting a wooden surface, followed by echoing silence
  fichier : scene_0.wav | 6.00s | RMS 0.0361 | centroide 1621 Hz | platitude 0.378



Scene 2 : Birds singing in a quiet forest with light wind
  fichier : scene_1.wav | 6.00s | RMS 0.0195 | centroide 2658 Hz | platitude 0.536



Scene 3 : A dog barks while somebody plays the trumpet in a busy street
  fichier : scene_2.wav | 6.00s | RMS 0.0178 | centroide 1205 Hz | platitude 0.086



Scene 4 : Rain falling on a metal roof with distant thunder
  fichier : scene_3.wav | 6.00s | RMS 0.0269 | centroide 2544 Hz | platitude 0.603


Les quatre scenes sont des instances **nouvelles** de leur classe -- pas des extraits recuperes. La lecture croisee des descripteurs et de l'ecoute repond a deux questions differentes : les descripteurs chiffrent l'energie et la repartition spectrale, l'ecoute juge la plausibilite de la scene. La composition multi-concepts (scene 3) est celle que le papier designe comme le test de generalisation : trois sources, des durees et des positions relatives differentes, une combinaison improbable dans les donnees d'entrainement.

### Interpretation : generation de scenes

| Aspect | Valeur typique | Signification |
|---|---|---|
| RTF (GPU) | inferieur a 1 | plus rapide que le temps reel : la generation 6 s prend moins de 6 s |
| RTF (CPU) | superieur a 1, souvent > 10 | le decodeur autoregressif paie chaque token sequentiellement |
| Pic VRAM | quelques GiB | le decodeur en demi-precision tient sur un GPU de bureau 8 Go |
| RMS / centroide | varies par scene | a comparer d'une scene a l'autre : une scene mixte devrait etaler le spectre (centroide plus haute) la ou une texture douce le concentre |

Le contraste structurant est **RTF vs composition** : la diffusion de 06-1 produit 5 s d'audio en ~0,5 s sur une RTX 3090, mais decroche sur les transitoires ; l'autoregressif paie sa latence en echange d'une adherence au texte plus docile sur les scenes multi-concepts. La section 4 chiffre ce partage.

### Exercice 1 : Descriptions precises vs vagues pour un bruitage

**Duree estimee :** 15 minutes

Comme pour MusicGen, la qualite depend de la precision de la description. Objectif : pour un meme evenement (un orage), rediger 3 descriptions de precision croissante, generer l'audio de chacune, et comparer les descripteurs.

**Indices** :
- `# Etape 1` : Ecrire 3 descriptions : "thunder", "distant thunder rumble", "Distant thunder rumbling across a valley, rain starting on leaves, wind rising"
- `# Etape 2` : Generer chaque description avec les memes parametres (`temperature=1.0`, `duration=6`)
- `# Etape 3` : Calculer RMS et centroide de chaque audio avec `audio_descriptors`
- `# Indice` : Une description plus riche devrait etaler le spectre (centroide et platitude differentes) et stabiliser l'energie

In [7]:
def compare_description_precision(descriptions, model, output_dir):
    """
    Compare la generation selon la precision de la description.

    Args:
        descriptions (list): descriptions de precision croissante
        model: modele AudioGen charge
        output_dir (Path): repertoire de sauvegarde

    Returns:
        list: liste de dictionnaires avec 'description', 'rms', 'centroid_hz', 'flatness'
    """
    # TODO etudiant : implementer la comparaison
    # Etape 1 : generer l'audio de chaque description avec des parametres identiques
    # Etape 2 : calculer les descripteurs de chaque signal (audio_descriptors)
    # Etape 3 : sauvegarder et collecter les metriques
    results = []
    return results  # TODO etudiant : retourner les resultats

# TODO etudiant : tester avec 3 descriptions de precision croissante
# storm_descriptions = [
#     "thunder",
#     "distant thunder rumble",
#     "Distant thunder rumbling across a valley, rain starting on leaves, wind rising",
# ]
# desc_results = compare_description_precision(storm_descriptions, model, OUTPUT_DIR)
# for r in desc_results:
#     print(f"Desc: {r['description'][:42]:<44} RMS: {r['rms']:.4f}  Centroide: {r['centroid_hz']:.0f} Hz")
print("Exercice a completer")


Exercice a completer


## Section 3 : Temperature, top_k et la diversite mesurable

Le decodeur echantillonne la sequence de tokens : `temperature` aplatit ou durcit la distribution, `top_k` restreint l'echantillonnage aux k tokens les plus probables, `cfg_coef` pese le conditionnement texte contre la distribution inconditionnelle (guidance sans classifieur, gamma = 3,0 dans le papier). Ce sont les trois leviers d'une generation AR -- les memes roles que `num_inference_steps`, prompt negatif et `audio_length_in_s` pour AudioLDM dans 06-1.

Le protocole mesure la **diversite entre deux generations du meme prompt** : pour chaque valeur du parametre, deux tirages, puis l'ecart des descripteurs entre les deux clips. L'attente theorique est simple -- une temperature plus haute devrait ecarter les tirages, un top_k plus bas les rapprocher -- mais deux tirages ne suffisent pas a l'etablir : l'ecart mesure sur deux clips est bruite, et les valeurs de la table se lisent comme une indication a reproduire avec plus d'echantillons (exercice `measure_generation_diversity`). Le tableau affiche donc aussi le RMS de chaque tirage : une valeur tres basse signale un echantillonnage qui s'est effondre sur une continuation quasi silencieuse, ce que la temperature basse rend possible.

In [8]:
# Balayage temperature et top_k, diversite mesuree
print("BALAYAGE TEMPERATURE / TOP_K")
print("=" * 45)

sweep_prompt = "Wind chimes moving in a light breeze, with occasional soft bird calls"

def sweep_param(param_name, values, n_samples=2):
    """Pour chaque valeur : n_samples tirages, descripteurs + ecart entre tirages."""
    rows = []
    for val in values:
        kwargs = {param_name: val}
        base = dict(duration=duration_seconds, cfg_coef=cfg_coef)
        base.update(kwargs)
        model.set_generation_params(**base)
        t0 = time.time()
        outs = model.generate([sweep_prompt] * n_samples)
        dt = (time.time() - t0) / n_samples
        descs = [audio_descriptors(o.detach().cpu().float().numpy().squeeze(), model.sample_rate)
                 for o in outs]
        delta_rms = abs(descs[0]["rms"] - descs[1]["rms"])
        delta_cent = abs(descs[0]["centroid_hz"] - descs[1]["centroid_hz"])
        rows.append({
            param_name: val, "gen_s": dt,
            "rms": descs[0]["rms"], "centroid_hz": descs[0]["centroid_hz"],
            "flatness": descs[0]["flatness"],
            "delta_rms": delta_rms, "delta_centroid_hz": delta_cent,
        })
        print(f"  {param_name}={val} : {dt:.1f}s/tirage | RMS {descs[0]['rms']:.4f} | "
              f"centroide {descs[0]['centroid_hz']:.0f} Hz | ecart_tirages "
              f"dRMS {delta_rms:.4f} / dCent {delta_cent:.0f} Hz")
    return rows

sweep_rows = []
if audiogen_loaded and generate_audio:
    temps = [0.5, 1.0, 1.5]
    print(f"\nTemperature sur : {sweep_prompt[:60]}...")
    sweep_rows += sweep_param("temperature", temps)
    ks = [50, 250, 500] if sweep_full else [50, 250]
    print(f"\nTop_k :")
    sweep_rows += sweep_param("top_k", ks)
    print("\nDiversite = ecart de descripteurs ENTRE les 2 tirages du meme prompt")
else:
    print("Balayage saute : modele non charge")


BALAYAGE TEMPERATURE / TOP_K

Temperature sur : Wind chimes moving in a light breeze, with occasional soft b...


  temperature=0.5 : 10.1s/tirage | RMS 0.0131 | centroide 4358 Hz | ecart_tirages dRMS 0.0125 / dCent 2548 Hz


  temperature=1.0 : 9.3s/tirage | RMS 0.0405 | centroide 3481 Hz | ecart_tirages dRMS 0.0112 / dCent 862 Hz


  temperature=1.5 : 9.8s/tirage | RMS 0.0409 | centroide 1895 Hz | ecart_tirages dRMS 0.0117 / dCent 1240 Hz

Top_k :


  top_k=50 : 10.5s/tirage | RMS 0.0353 | centroide 4296 Hz | ecart_tirages dRMS 0.0238 / dCent 896 Hz


  top_k=250 : 9.8s/tirage | RMS 0.0418 | centroide 4325 Hz | ecart_tirages dRMS 0.0363 / dCent 1585 Hz


  top_k=500 : 9.8s/tirage | RMS 0.0297 | centroide 3258 Hz | ecart_tirages dRMS 0.0388 / dCent 880 Hz

Diversite = ecart de descripteurs ENTRE les 2 tirages du meme prompt


### Interpretation : le balayage

La ligne qui compte est l'**ecart entre tirages** : il chiffre la diversite que le parametre achete (ou retire). L'attente est que la temperature ecarte les tirages vers les choix improbables -- transitoires plus surprenants -- et que le top_k les rapproche en troncant la queue de la distribution. Avec deux tirages par valeur, cette direction se lit comme une tendance a confirmer, pas comme une loi : c'est precisement l'objet de l'exercice suivant. La platitude spectrale dit si le bruit gagne du terrain (valeur qui monte = spectre plus blanc = artefacts probables), et le RMS dit si l'echantillonnage a produit du son ou s'est effondre sur du silence.

### Exercice 2 : Diversite entre graines sur un meme prompt

**Duree estimee :** 15 minutes

Le balayage ci-dessus compare 2 tirages par valeur. Objectif : pour `temperature=1.0` fixe, generer **N = 4** tirages du meme prompt, calculer les descripteurs de chacun, et rapporter moyenne et ecart-type de chaque descripteur -- c'est la dispersion inter-echantillons du modele.

**Indices** :
- `# Etape 1` : appeler `model.generate([prompt] * 4)` (un appel, quatre tirages)
- `# Etape 2` : collecter `audio_descriptors` pour chaque sortie
- `# Etape 3` : rapporter moyenne et ecart-type (`np.mean`, `np.std`) par descripteur
- `# Indice` : comparer l'ecart-type du centroide a l'ecart entre deux prompts differents de la section 2 -- intra-prompt vs inter-prompts

In [9]:
def measure_generation_diversity(prompt, n_samples, model):
    """
    Mesure la dispersion des descripteurs sur n_samples tirages du meme prompt.

    Args:
        prompt (str): description de la scene
        n_samples (int): nombre de tirages
        model: modele AudioGen charge

    Returns:
        dict: moyenne et ecart-type de chaque descripteur, ou None
    """
    # TODO etudiant : implementer la mesure de dispersion
    # Etape 1 : generer n_samples tirages du meme prompt
    # Etape 2 : calculer audio_descriptors pour chaque tirage
    # Etape 3 : agreger moyenne / ecart-type par descripteur
    result = None  # TODO etudiant : remplacer par le dictionnaire agrege
    return result

# TODO etudiant : mesurer puis comparer a la section 2
# diversity = measure_generation_diversity("Waves crashing on a rocky shore", 4, model)
# for k, (m, s) in diversity.items():
#     print(f"{k}: {m:.4f} +- {s:.4f}")
print("Exercice a completer")


Exercice a completer


## Section 4 : Positionnement - autoregressif vs diffusion vs musique

Les trois carnets de la serie qui generent de l'audio depuis du texte se rangent sur une ligne : **AudioLDM-S** (06-1, diffusion latente + vocodeur, 421 M, AudioCaps), **AudioGen** (ce carnet, autoregressif sur tokens de codec, taille publiee ~1,5 Md, 10 jeux de donnees), **MusicGen** (02-3, le meme schema AR qu'AudioGen mais cible musique, 300 M a 6,6 Md selon la variante). Le tableau croise les mesures **commitees** de chacun -- materiel different, ordres de grandeur comparables.

Le carnet de 06-1 mesure la variante **AudioLDM-S** ; **AudioLDM 2** est un modele distinct, plus recent, traite par le carnet d'effets sonores `Audio-02-10-Effets-Sonores-AudioLDM2-Python` de cette sous-serie. Ces deux carnets produisent des bruitages par diffusion ; celui-ci mesure le meme objectif par voie autoregressive.

In [10]:
# Tableau croise des trois generateurs (mesures commitees)
print("POSITIONNEMENT DES TROIS GENERATEURS")
print("=" * 45)

if audiogen_loaded and generation_stats:
    rtf_here = None
    print('''
| Generateur | Paradigme | Parametres | Materiel (run commite) | Latence / RTF | Point fort documente |
|---|---|---|---|---|---|
| AudioLDM-S (06-1) | diffusion latente + vocodeur | 421 M | RTX 3090 24 Go | 0,52 s pour 5,12 s (RTF ~0,10) | vitesse, textures ; decroche sur transitoires (mel ~41 dB) |
| AudioGen-medium (ici) | autoregressif sur tokens | ~1,5 Md | voir cellule de chargement ci-dessus | voir RTF section 2 | composition multi-concepts, adherence texte |
| MusicGen-small (02-3) | autoregressif sur tokens (musique) | 300 M | RTX 4060 8 Go | ~7 s pour 8 s de musique | melodies instrumentales, conditionnement chromatique |
''')
    print("Lecture croisee :")
    print("- 06-1 mesure 0,52 s pour 5,12 s d'AudioLDM-S sur RTX 3090 (RTF ~0,10).")
    print("- 02-3 mesure ~7 s pour 8 s de musique avec musicgen-small sur RTX 4060.")
    print("- Ce carnet mesure le RTF AudioGen ci-dessus, sur le materiel affiche en section 1.")
    print("- Le papier AudioGen (Kreuk et al. 2023, Table 1) : FAD 1,82 / REL 68,73 pour 1 Md")
    print("  contre FAD 7,39 / REL 55,91 pour DiffSound 400 M -- l'AR achete l'adherence texte.")
else:
    print("Tableau saute : modele non charge")


POSITIONNEMENT DES TROIS GENERATEURS

| Generateur | Paradigme | Parametres | Materiel (run commite) | Latence / RTF | Point fort documente |
|---|---|---|---|---|---|
| AudioLDM-S (06-1) | diffusion latente + vocodeur | 421 M | RTX 3090 24 Go | 0,52 s pour 5,12 s (RTF ~0,10) | vitesse, textures ; decroche sur transitoires (mel ~41 dB) |
| AudioGen-medium (ici) | autoregressif sur tokens | ~1,5 Md | voir cellule de chargement ci-dessus | voir RTF section 2 | composition multi-concepts, adherence texte |
| MusicGen-small (02-3) | autoregressif sur tokens (musique) | 300 M | RTX 4060 8 Go | ~7 s pour 8 s de musique | melodies instrumentales, conditionnement chromatique |

Lecture croisee :
- 06-1 mesure 0,52 s pour 5,12 s d'AudioLDM-S sur RTX 3090 (RTF ~0,10).
- 02-3 mesure ~7 s pour 8 s de musique avec musicgen-small sur RTX 4060.
- Ce carnet mesure le RTF AudioGen ci-dessus, sur le materiel affiche en section 1.
- Le papier AudioGen (Kreuk et al. 2023, Table 1) : FAD 1,82 / REL 68,73 p

### Interpretation : quand l'autoregressif bat la diffusion

**Sur la latence brute, la diffusion gagne** : AudioLDM-S produit 5 s d'audio en une demi-seconde sur un GPU de 24 Go, et les pas de diffusion se parallelisent -- l'autoregressif, lui, ecrit chaque token sequentiellement et son RTF descend rarement tres bas. **Sur la composition, l'autoregressif gagne** : la scene trois (chien + trompette + rue) demande une generalisation combinatoire que le papier mesure (REL 68,73 vs 55,91 pour DiffSound) et que l'ecoute confirme -- le conditionnement T5 pre-entraine transfere des concepts textuels absents des paires texte-audio d'entrainement. Le partage est donc : **texture rapide et mono-source -> diffusion ; scene multi-concepts decrite en langue -> autoregressif**. MusicGen ferme la ligne : meme moteur qu'AudioGen, domaine musique, et l'echelle de parametres y joue le role que la diversite des donnees joue ici.

### Exercice 3 : Le guidage sans classifieur (cfg_coef)

**Duree estimee :** 15 minutes

`cfg_coef` est le gamma du papier (3,0 par defaut) : il pese la distribution conditionnee au texte contre la distribution inconditionnelle. Objectif : balayer `cfg_coef` sur {1, 3, 5} pour un meme prompt, generer un tirage par valeur, et ecouter le compromis adherence/diversite.

**Indices** :
- `# Etape 1` : pour chaque valeur, `model.set_generation_params(duration=6, cfg_coef=val)` puis `model.generate([prompt])`
- `# Etape 2` : sauvegarder chaque tirage et calculer `audio_descriptors`
- `# Etape 3` : comparer les descripteurs et ecouter -- a gamma eleve la scene colle au texte mais peut durcir (artefacts)
- `# Indice` : reutiliser la fonction `sweep_param` de la section 3 avec `sweep_param("cfg_coef", [1, 3, 5], n_samples=1)`

In [11]:
def explore_cfg_coef(prompt, cfg_values, model, output_dir):
    """
    Teste l'impact du coefficient de guidance sur la generation.

    Args:
        prompt (str): description de la scene
        cfg_values (list): valeurs de cfg_coef a tester
        model: modele AudioGen charge
        output_dir (Path): repertoire de sauvegarde

    Returns:
        list: une entree par valeur avec descripteurs, ou liste vide
    """
    # TODO etudiant : implementer le balayage
    # Etape 1 : pour chaque cfg_coef, generer un tirage du prompt
    # Etape 2 : sauvegarder le WAV et calculer les descripteurs
    # Etape 3 : collecter et retourner les resultats
    results = []
    return results  # TODO etudiant : retourner les resultats

# TODO etudiant : tester puis ecouter
# cfg_results = explore_cfg_coef("A church bell tolling in a small town square", [1, 3, 5], model, OUTPUT_DIR)
# for r in cfg_results:
#     print(f"cfg={r['cfg_coef']} : RMS {r['rms']:.4f} | centroide {r['centroid_hz']:.0f} Hz")
print("Exercice a completer")


Exercice a completer


## Bonnes pratiques : rediger une description de scene sonore

| Element | Exemple faible | Exemple fort |
|---|---|---|
| Source principale | "noise" | "a hammer hitting a wooden surface" |
| Contexte acoustique | "outside" | "in a quiet forest with light wind" |
| Concurrency | "sounds" | "a dog barks while somebody plays the trumpet" |
| Dynamique | "loud" | "followed by echoing silence" |

Une bonne description nomme les sources, leur position relative (fond/avant-plan) et la dynamique temporelle -- exactement les trois axes que l'augmentation par melange du papier entraine le modele a separer.

Toutes les generations sont terminees. La cellule suivante recapitule la session : fichiers produits, temps total, RTF et memoire -- les nombres a citer pour reproduire le protocole.

In [12]:
# Statistiques de session et prochaines etapes
print("STATISTIQUES DE SESSION")
print("=" * 45)

if generation_stats:
    total_gen = len(generation_stats) + (len(sweep_rows) if isinstance(sweep_rows, list) else 0)
    print(f"Scenes generees : {len(generation_stats)} + {len(sweep_rows)} tirages de balayage")
    print(f"Repertoire : GenAI/{OUTPUT_DIR.relative_to(GENAI_ROOT)}")
    for s in generation_stats:
        print(f"  {s['fichier']} : {s['prompt'][:55]}")
    if audiogen_loaded and device == "cuda" and torch.cuda.is_available():
        print(f"Pic VRAM session : {torch.cuda.max_memory_allocated() / (1024**3):.2f} GiB")
else:
    print("Aucune generation cette session")

print("\nProchaines etapes :")
print("- [MusicGen](02-3-MusicGen-Generation.ipynb) : le meme moteur AR, domaine musique")
print("- [AudioLDM](../06-Diffusion-SOTA/06-1-AudioLDM-SOTA-Comparison.ipynb) : le paradigme diffusion, mesure au meme protocole")
print("- [Demucs](02-4-Demucs-Source-Separation.ipynb) : l'operation inverse -- separer les sources d'un melange")


STATISTIQUES DE SESSION
Scenes generees : 4 + 6 tirages de balayage
Repertoire : GenAI/outputs\audio\audiogen
  scene_0.wav : A hammer hitting a wooden surface, followed by echoing 
  scene_1.wav : Birds singing in a quiet forest with light wind
  scene_2.wav : A dog barks while somebody plays the trumpet in a busy 
  scene_3.wav : Rain falling on a metal roof with distant thunder
Pic VRAM session : 5.66 GiB

Prochaines etapes :
- [MusicGen](02-3-MusicGen-Generation.ipynb) : le meme moteur AR, domaine musique
- [AudioLDM](../06-Diffusion-SOTA/06-1-AudioLDM-SOTA-Comparison.ipynb) : le paradigme diffusion, mesure au meme protocole
- [Demucs](02-4-Demucs-Source-Separation.ipynb) : l'operation inverse -- separer les sources d'un melange


## Synthese de la session AudioGen

### Architecture technique

| Composant | Role | Impact sur la generation |
|---|---|---|
| Codec neural (EnCodec) | compresse l'audio 16 kHz en tokens discrets | la qualite du codec plafonne la fidelite, pas le LM |
| Decodeur autoregressif (~1,5 Md) | ecrit la sequence de tokens conditionnee par T5 | chaque seconde d'audio est sequentielle : RTF borne par la longueur |
| Encodeur T5 pre-entraine | represente le prompt en langue | transfere des concepts textuels absents des donnees audio |

### Ce que la session a mesure

- **RTF et pic VRAM** du modele publie sur le materiel affiche en section 1 -- deux nombres directement comparables au protocole de 06-1 ;
- **l'effet de temperature et de top_k sur la diversite**, chiffre par l'ecart de descripteurs entre tirages d'un meme prompt ;
- **la composition multi-concepts** sur la scene empruntee au papier, la ou l'autoregressif achete son adherence au texte.

### References

- Kreuk, F., Synnaeve, G., Polyak, A., Singer, U., Defossez, A., Copet, J., Parikh, D., Taigman, Y., Adi, Y. (2023). *AudioGen: Textually Guided Audio Generation*. ICLR 2023 (arXiv:2209.15352).
- Audiocraft, la librairie Meta qui sert ce modele : https://github.com/facebookresearch/audiocraft
- Les poids `facebook/audiogen-medium` sont soumis a leur propre licence sur Hugging Face.

***

## CHALLENGE BONUS - Soundscape pour une scene de jeu video

**Points : 0.5 pts**

### Objectif

Produire une **soundscape en trois couches** pour une scene de jeu (exemple : "un camp de bivouac la nuit") : une couche de fond (feu qui craque), un evenement recurrent (un loup au loin), une texture atmospherique (vent dans les arbres) -- une generation par couche, puis un mixage a gain controle des trois WAV.

### Contraintes

1. Trois generations separees, `duration=6`, memes parametres de sampling ;
2. Mixage numpy : somme ponderee (ex. fond 0.5, evenement 0.3, texture 0.4), normalisation a -1/+1 ;
3. Comparer le mix a une generation directe du prompt complet -- que perd et que gagne le decoupage en couches ?